# Experiment 5 — Evaluating Extended Technical Features for SPY Direction and Downside Prediction

**Stock Forecasting Project**

Copyright © 2026 by Tien Le


> **Public Demo Version**
>
> This notebook preserves the original data workflow, model names, training, evaluation, results, and conclusions.
> Only proprietary feature-building formulas and original feature names are omitted where applicable.


# Table of Contents

1. [Introduction](#Introduction)
2. [Motivation](#Motivation)
3. [Research Questions](#Research-Questions)
4. [Dataset Description](#Dataset-Description)
5. [Methodology](#Methodology)
6. [Data Preparation](#Data-Preparation)
7. [Extended Feature Engineering](#Extended-Feature-Engineering)
8. [Feature Count](#Feature-Count)
9. [Avoiding Data Leakage](#Avoiding-Data-Leakage)
10. [Version 1: Next-Day Direction](#Version-1-Next-Day-Direction)
11. [Version 2: Large Downside Moves](#Version-2-Large-Downside-Moves)
12. [Comparison with Experiment 4](#Comparison-with-Experiment-4)
13. [Results](#Results)
14. [Discussion](#Discussion)
15. [Limitations](#Limitations)
16. [Final Conclusion](#Final-Conclusion)
17. [Transition to the Final Production System](#Transition-to-the-Final-Production-System)


# Introduction

Experiment 4 produced the strongest results in the project by using a compact set of 253 SPY candlestick, price-action, volume, trend, momentum, and volatility features.

Experiment 5 tests whether a more detailed and complicated feature representation can improve those results. The goal is to determine whether the additional predictors create useful information or simply add redundancy and noise.


# Motivation

Adding more features can improve performance when the new variables capture information missing from the original dataset. However, a larger feature set can also increase noise, redundancy, overfitting, computational cost, and difficulty of interpretation.

This experiment directly compares an extended technical-analysis feature system with the simpler compressed system from Experiment 4.


# Research Questions

> Do extended and more detailed technical-analysis features improve SPY direction and large-downside prediction compared with the simpler compressed feature system?

The experiment also asks:

1. How many predictors are created by the extended pipeline?
2. Does the extended system improve ordinary Up/Down prediction?
3. Does it improve recall and precision for large downside moves?
4. Do more complex models benefit from the larger feature set?
5. Is the additional complexity justified by the test results?


# Dataset Description

The analysis uses daily SPY market data downloaded from Yahoo Finance beginning on January 1, 2017.

After rolling calculations and higher-timeframe features are created, the usable feature period begins on July 31, 2019. The most recent 252 trading days, approximately one year, are reserved for testing.


# Methodology

The extended feature matrix is evaluated on two tasks:

1. Predict whether SPY will close Up or Down on the following trading day.
2. Predict whether tomorrow's return falls below several downside thresholds.

The models compared are Logistic Regression, Decision Tree, Random Forest, and Simple XGBoost.

For the downside-warning models, recall and precision are emphasized because accuracy alone can be misleading for rare-event targets.


# Data Preparation


In [1]:
# Packages
%pip install xgboost
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report)
import warnings
warnings.filterwarnings("ignore")
from IPython.display import Markdown, display

Note: you may need to restart the kernel to use updated packages.


## Importing data from Yahoo Finance

In [2]:
# No public data-download package is required.
# The private preparation notebook creates the reproducible demo snapshot.


In [3]:
# Live download moved to PRIVATE-Prepare-Demo-Data.ipynb.


## Choosing the Historical Data Window for Prediction
When predicting tomorrow's SPY closing price, using more historical data is not always better. In many cases, 20 years of data does not provide significantly better predictions than 5–10 years of data. In fact, older data can sometimes reduce performance because market conditions, trading behavior, regulations, and economic environments change over time.

For this reason, many quantitative trading strategies are developed using approximately 3–10 years of historical data rather than the entire available history.

### Why I Chose a Start Date of 2017-01-01
I selected a start date of 2017-01-01, which provides approximately 9.5 years of market data. This period captures several important market regimes and major economic events, including:

- Trump's first presidential term (2017–2021)
- The COVID-19 market crash and recovery
- The Biden administration years
- The 2022 bear market
- The AI-driven bull market beginning in 2023
- Trump's second presidential term beginning in 2025

This time period is long enough to include multiple market environments while remaining recent enough to be relevant for forecasting current market behavior.

# Extended Feature Engineering

The extended feature pipeline creates a more detailed representation of SPY chart structure than Experiment 4.

The predictors include current and lagged candlestick structure, rolling returns, rolling volatility, trend, momentum, range position, volume behavior, streaks, weekly candles, monthly candles, multi-timeframe relationships, and additional transformations.

All predictors are calculated using information available by the close of the current trading day.


# Feature Engineering Strategy

## Overview

The objective of this project is to construct features that imitate how human technical traders analyze stock charts instead of using raw stock prices directly.

The original market data contain the standard OHLCV information:

- Open
- High
- Low
- Close
- Volume

These values are transformed into meaningful descriptions of market behavior, including candlestick shape, trend, momentum, volatility, volume activity, and higher-timeframe chart structure.

In addition to the current daily candle, the feature set also includes completed weekly and monthly candles. This gives the machine learning models short-term, medium-term, and long-term market context.

The feature engineering process uses only information available at the close of each trading day when predicting the following trading day, preventing look-ahead bias and data leakage.

---

## Feature Engineering Pipeline

```text
Download SPY OHLCV Data
          |
          v
Clean Yahoo Finance Data
          |
          v
Create Daily Candlestick Features
          |
          +--> Weekly Candlestick Features
          |
          +--> Monthly Candlestick Features
          |
          +--> Trend Features
          |
          +--> Volume Features
          |
          +--> Momentum Indicators
          |
          +--> Volatility Features
          |
          v
Combine All Features
          |
          v
Remove Missing and Invalid Values
          |
          v
Return Feature Matrix X
```

---

## 1. Data Cleaning

The Yahoo Finance data are cleaned before feature construction.

The cleaning process includes:

- Flatten Yahoo Finance MultiIndex columns when necessary.
- Preserve the DateTime index.
- Sort observations chronologically.
- Replace infinite values with missing values.
- Remove rows that do not have sufficient historical data for rolling calculations.

---

## 2. Daily Candlestick Features

Using the daily OHLCV data:

- Open
- High
- Low
- Close
- Volume

compute:

- Daily Return
- Body Percentage
- Absolute Body Percentage
- Upper Shadow Percentage
- Lower Shadow Percentage
- Close Position
- Gap Percentage
- Range Percentage
- Bull Candle
- Bear Candle
- Doji

These features describe the daily candlestick without relying on raw stock prices.

---

## 3. Volume Features

Transform raw trading volume into relative measures:

- Volume Change
- Volume Ratio (20-day)
- Volume Z-Score (20-day)
- Dollar Volume Ratio
- Up-Volume Ratio
- Down-Volume Ratio
- OBV Change (5-day)
- OBV Change (20-day)

These features measure whether price movement is supported by unusually high or low trading activity.

---

## 4. Trend Structure

Create trend structure features including:

- Higher High
- Higher Low
- Lower High
- Lower Low

Also compute consecutive streaks:

- Consecutive Up Days
- Consecutive Down Days
- Bull Candle Streak
- Bear Candle Streak
- Higher High Streak
- Higher Low Streak
- Lower High Streak
- Lower Low Streak

These describe persistent market behavior.

---

## 5. Rolling Candlestick Summaries

For rolling windows of:

- 5 days
- 10 days
- 20 days
- 30 days
- 60 days

compute:

- Total Return
- Return Volatility
- Bull Candle Ratio
- Bear Candle Ratio
- Doji Ratio
- Mean Body
- Mean Absolute Body
- Mean Upper Shadow
- Mean Lower Shadow
- Mean Close Position
- Mean Range
- Maximum Body
- Maximum Range
- Mean Volume Ratio
- Maximum Volume Ratio
- Higher High Ratio
- Higher Low Ratio
- Lower High Ratio
- Lower Low Ratio
- Distance from Rolling High
- Distance from Rolling Low

---

## 6. Trend Indicators

Compute moving averages over:

- 5 days
- 10 days
- 20 days
- 50 days
- 100 days
- 200 days

Create:

- Close vs SMA
- SMA Slope
- SMA5 vs SMA20
- SMA20 vs SMA50
- SMA50 vs SMA200

These features describe both short-term and long-term market trends.

---

## 7. Momentum Indicators

Compute:

- RSI (14)
- MACD
- MACD Signal
- MACD Histogram
- Stochastic %K
- Stochastic %D
- Williams %R
- Today's Return
- Return Lag 1
- Return Lag 2
- Return Lag 3
- Return Lag 4

These summarize buying and selling momentum.

---

## 8. Volatility Features

Compute:

- ATR Percentage
- Rolling Volatility (5-day)
- Rolling Volatility (10-day)
- Rolling Volatility (20-day)
- Bollinger Band Position
- Bollinger Band Width
- Bollinger Band Width Change
- Distance from Recent Highs
- Distance from Recent Lows

These describe market expansion and contraction.

---

## 9. Weekly Candlestick Context

Resample daily OHLCV data into completed weekly candles.

For each completed week, compute features similar to the daily candle:

- Weekly Return
- Weekly Body
- Weekly Shadows
- Weekly Range
- Weekly Close Position
- Weekly Volume Features
- Weekly Trend Structure
- Weekly Rolling Summaries

Shift weekly features by one completed week before merging with daily data.

---

## 10. Monthly Candlestick Context

Resample daily OHLCV data into completed monthly candles.

For each completed month, compute:

- Monthly Return
- Monthly Body
- Monthly Shadows
- Monthly Range
- Monthly Close Position
- Monthly Volume Features
- Monthly Trend Structure
- Monthly Rolling Summaries

Shift monthly features by one completed month before merging with daily data.

---

## 11. Final Feature Matrix

Combine:

- Daily Features
- Weekly Features
- Monthly Features
- Trend Features
- Volume Features
- Momentum Features
- Volatility Features

After combining:

- Replace infinite values with missing values.
- Remove incomplete rows.
- Return only the feature matrix **X**.

Targets are created separately for each experiment to prevent target leakage.

---

## Advantages of This Feature Engineering Strategy

- Uses Open, High, Low, Close, and Volume information.
- Mimics traditional candlestick chart analysis.
- Uses relative features instead of raw prices.
- Includes daily, weekly, and monthly market context.
- Captures trend, momentum, volatility, and volume behavior.
- Avoids target leakage.
- Avoids look-ahead bias from incomplete weekly or monthly candles.
- Produces one reusable feature matrix for multiple prediction tasks.
- Allows fair comparison across Decision Tree, Logistic Regression, Random Forest, and XGBoost models.

In [4]:
# Public-demo feature loader
# The private feature formulas are intentionally not included.
from pathlib import Path

def locate_demo_file(filename):
    for path in [Path(filename), Path("data") / filename]:
        if path.exists():
            return path
    raise FileNotFoundError(
        f"{filename} was not found. Run PRIVATE-Prepare-Demo-Data.ipynb, "
        "then place the generated CSV beside this notebook or in data/."
    )

def load_private_feature_snapshot(filename):
    snapshot = pd.read_csv(
        locate_demo_file(filename),
        parse_dates=["Date"]
    ).set_index("Date").sort_index()
    required = {"Target", "Next_Day_Return", "Market_Close"}
    missing = required.difference(snapshot.columns)
    if missing:
        raise ValueError(f"Missing snapshot columns: {sorted(missing)}")
    feature_cols = [c for c in snapshot if c.startswith("Feature_")]
    if not feature_cols:
        raise ValueError("No anonymized Feature_ columns were found.")
    return snapshot, feature_cols

snapshot, feature_cols = load_private_feature_snapshot("demo_features_experiment5.csv")
X = snapshot[feature_cols].copy()
print("Feature matrix created successfully.")
print("Shape:", X.shape)
print("Number of features:", X.shape[1])
print("Feature period:", X.index.min().date(), "to", X.index.max().date())


Feature matrix created successfully.
Shape: (1745, 325)
Number of features: 325
Feature period: 2019-07-31 to 2026-07-10


# Feature Count

The extended feature-engineering pipeline produces:

## **325 predictors**

The resulting feature matrix contains:

- **1,744 observations**
- **325 engineered features**
- Feature period: **July 31, 2019 through July 9, 2026**
- No target column inside the predictor matrix

Experiment 4 used **253 predictors**, so Experiment 5 adds:

\[
325 - 253 = 72
\]

## **72 additional features**

This is approximately a **28.5% increase** in the number of predictors.

The code above prints the matrix shape and exact feature count whenever the notebook is rerun.


# Avoiding Data Leakage

The experiment prevents leakage by using only current and historical information for all features, creating the future-return target afterward, dropping the final unknown target row, using a chronological split, computing downside thresholds from training data only, and excluding the target from the predictor matrix.


# Version 1: Next-Day Direction

The first model predicts whether SPY will close Up or Down on the following trading day using all 325 predictors.


## Predict Up/Down for SP500 tomorrow using Decision Tree

In [5]:
# =====================================================
# Decision Tree: Predict Tomorrow Up or Down
# =====================================================


# -----------------------------
# 1. Start with the feature matrix
# -----------------------------

# X was created previously using:
# X = make_spy_candle_structure_features(spy)

X_direction = X.copy()

print("Original feature shape:", X_direction.shape)
print("Target included in X:", "Target" in X_direction.columns)


# -----------------------------
# 2. Create next-day return
# -----------------------------

future_return = snapshot["Next_Day_Return"].copy()


# -----------------------------
# 3. Align X with future returns
# -----------------------------

common_index = X_direction.index.intersection(
    future_return.dropna().index
)

X_direction = X_direction.loc[common_index].copy()
future_return = future_return.loc[common_index].copy()

# Remove invalid values
X_direction = X_direction.replace(
    [np.inf, -np.inf],
    np.nan
)

valid_rows = (
    X_direction.notna().all(axis=1)
    & future_return.notna()
)

X_direction = X_direction.loc[valid_rows].copy()
future_return = future_return.loc[valid_rows].copy()


# -----------------------------
# 4. Create Up/Down target
# -----------------------------

# 1 = Tomorrow closes higher
# 0 = Tomorrow closes equal or lower

y = (
    future_return > 0
).astype(int)

print("\nTarget distribution:")
print(y.value_counts())

print("\nPositive rate:")
print(y.mean())


# -----------------------------
# 5. Chronological train/test split
# Use most recent year as test data
# -----------------------------

test_start = (
    X_direction.index.max()
    - pd.DateOffset(years=1)
)

X_train = X_direction.loc[
    X_direction.index < test_start
].copy()

X_test = X_direction.loc[
    X_direction.index >= test_start
].copy()

y_train = y.loc[X_train.index].copy()
y_test = y.loc[X_test.index].copy()

print("\nTrain shape:", X_train.shape)
print("Test shape :", X_test.shape)

print(
    "\nTrain period:",
    X_train.index.min().date(),
    "to",
    X_train.index.max().date()
)

print(
    "Test period:",
    X_test.index.min().date(),
    "to",
    X_test.index.max().date()
)

print("\nTraining Up rate:", y_train.mean())
print("Testing Up rate :", y_test.mean())


# -----------------------------
# 6. Train Decision Tree
# -----------------------------

tree_structure = DecisionTreeClassifier(
    max_depth=3,
    min_samples_leaf=20,
    min_samples_split=5,
    random_state=42
)

tree_structure.fit(
    X_train,
    y_train
)


# -----------------------------
# 7. Predict
# -----------------------------

train_pred = tree_structure.predict(X_train)
test_pred = tree_structure.predict(X_test)


# -----------------------------
# 8. Evaluate
# -----------------------------

cm = confusion_matrix(
    y_test,
    test_pred,
    labels=[0, 1]
)

train_accuracy = accuracy_score(
    y_train,
    train_pred
)

test_accuracy = accuracy_score(
    y_test,
    test_pred
)

precision = precision_score(
    y_test,
    test_pred,
    zero_division=0
)

recall = recall_score(
    y_test,
    test_pred,
    zero_division=0
)

f1 = f1_score(
    y_test,
    test_pred,
    zero_division=0
)

always_up_accuracy = (
    y_test == 1
).mean()

always_down_accuracy = (
    y_test == 0
).mean()


# -----------------------------
# 9. Prediction counts
# -----------------------------

actual_up = int(
    (y_test == 1).sum()
)

predicted_up = int(
    (test_pred == 1).sum()
)

correct_up = int(
    cm[1, 1]
)

actual_down = int(
    (y_test == 0).sum()
)

predicted_down = int(
    (test_pred == 0).sum()
)

correct_down = int(
    cm[0, 0]
)


# -----------------------------
# 10. Print results
# -----------------------------

print(
    "\n========== Decision Tree: Tomorrow Up/Down =========="
)

print("Train Accuracy :", train_accuracy)
print("Test Accuracy  :", test_accuracy)

print(
    "Always-Up Accuracy:",
    always_up_accuracy
)

print(
    "Always-Down Accuracy:",
    always_down_accuracy
)

print("\nPrecision for Up:", precision)
print("Recall for Up   :", recall)
print("F1 Score for Up :", f1)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        test_pred,
        target_names=[
            "Down",
            "Up"
        ],
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(cm)

print("\nPrediction Summary:")

print("Actual Up Days    :", actual_up)
print("Predicted Up Days :", predicted_up)
print("Correct Up Days   :", correct_up)

print("\nActual Down Days    :", actual_down)
print("Predicted Down Days :", predicted_down)
print("Correct Down Days   :", correct_down)

print("\nObservation:")
print(
    f"The Decision Tree predicted {predicted_up} Up days "
    f"and {predicted_down} Down days. "
    f"It correctly identified {correct_up} of the "
    f"{actual_up} actual Up days and {correct_down} of the "
    f"{actual_down} actual Down days. "
    f"Test accuracy was {test_accuracy:.4f}, compared with "
    f"{always_up_accuracy:.4f} for always predicting Up."
)

Original feature shape: (1745, 325)
Target included in X: False

Target distribution:
Next_Day_Return
1    962
0    782
Name: count, dtype: int64

Positive rate:
0.551605504587156

Train shape: (1492, 325)
Test shape : (252, 325)

Train period: 2019-07-31 to 2025-07-08
Test period: 2025-07-09 to 2026-07-09

Training Up rate: 0.5516085790884718
Testing Up rate : 0.5515873015873016

========== Decision Tree: Tomorrow Up/Down ==========
Train Accuracy : 0.5971849865951743
Test Accuracy  : 0.5396825396825397
Always-Up Accuracy: 0.5515873015873016
Always-Down Accuracy: 0.44841269841269843

Precision for Up: 0.5477178423236515
Recall for Up   : 0.9496402877697842
F1 Score for Up : 0.6947368421052632

Classification Report:
              precision    recall  f1-score   support

        Down       0.36      0.04      0.06       113
          Up       0.55      0.95      0.69       139

    accuracy                           0.54       252
   macro avg       0.46      0.49      0.38       252
w

## Direction-Prediction Result

The Decision Tree achieved:

- **Train accuracy:** approximately 58.28%
- **Test accuracy:** approximately 53.97%
- **Always-Up baseline:** approximately 55.16%
- **Predicted Down days:** 11
- **Correct Down predictions:** 4

The extended feature system did not improve ordinary direction prediction. Test accuracy was below the Always-Up baseline, and the model identified only 4 of the 113 actual Down days.


# Version 2: Large Downside Moves

The second analysis evaluates multiple downside targets using Logistic Regression, Decision Tree, Random Forest, and Simple XGBoost. The models are ranked primarily by recall.


## Predict if the SP500 goes down big tomorrow

In [6]:
# =====================================================
# Optional XGBoost
# =====================================================

try:
    from xgboost import XGBClassifier
    has_xgb = True
except ImportError:
    has_xgb = False


# =====================================================
# 1. Prepare new features and next-day returns
# =====================================================

future_return = snapshot["Next_Day_Return"].copy()

# Use the new feature matrix
# It should already contain features only
base_X = X.drop(
    columns=["Target"],
    errors="ignore"
).copy()

print(
    "Target included as feature:",
    "Target" in base_X.columns
)

# Align features and next-day returns
common_index = base_X.index.intersection(
    future_return.dropna().index
)

base_X = base_X.loc[common_index].copy()
future_return = future_return.loc[common_index].copy()

# Remove invalid values
base_X = base_X.replace(
    [np.inf, -np.inf],
    np.nan
)

valid_rows = (
    base_X.notna().all(axis=1)
    & future_return.notna()
)

base_X = base_X.loc[valid_rows].copy()
future_return = future_return.loc[valid_rows].copy()

print("\nAvailable observations:", len(base_X))
print("Number of features:", base_X.shape[1])


# =====================================================
# 2. Fixed chronological train/test split
# Most recent calendar year is the test period
# =====================================================

test_start = (
    base_X.index.max()
    - pd.DateOffset(years=1)
)

X_train = base_X.loc[
    base_X.index < test_start
].copy()

X_test = base_X.loc[
    base_X.index >= test_start
].copy()

train_future_return = future_return.loc[
    X_train.index
].copy()

test_future_return = future_return.loc[
    X_test.index
].copy()

print("\nTrain shape:", X_train.shape)
print("Test shape :", X_test.shape)

print(
    "Test period:",
    X_test.index.min().date(),
    "to",
    X_test.index.max().date()
)


# =====================================================
# 3. Models to compare
# =====================================================

models = {
    "Decision Tree": DecisionTreeClassifier(
        max_depth=3,
        min_samples_leaf=20,
        min_samples_split=5,
        class_weight="balanced",
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=500,
        max_depth=5,
        min_samples_leaf=20,
        min_samples_split=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    # StandardScaler is fitted only on X_train
    "Logistic Regression": Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            LogisticRegression(
                max_iter=5000,
                class_weight="balanced",
                random_state=42
            )
        )
    ])
}

if has_xgb:
    models["Simple XGBoost"] = XGBClassifier(
        objective="binary:logistic",
        n_estimators=100,
        learning_rate=0.03,
        max_depth=2,
        subsample=0.7,
        colsample_bytree=0.7,
        min_child_weight=10,
        gamma=1,
        reg_alpha=1,
        reg_lambda=10,
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1
    )


# =====================================================
# 4. Target definitions
# =====================================================

target_settings = {
    "Top 5% Down": 0.05,
    "Top 10% Down": 0.10,
    "Top 15% Down": 0.15,
    "Top 20% Down": 0.20,
    "Top 25% Down": 0.25,
    "Top 30% Down": 0.30,
    "Top 35% Down": 0.35,
    "Top 40% Down": 0.40,
    "Top 45% Down": 0.45,
    "All Down Days": "all_down"
}


# =====================================================
# 5. Run all target and model experiments
# =====================================================

results = []

for target_name, q in target_settings.items():

    # -------------------------------------------------
    # Create targets
    # Percentile threshold uses TRAINING returns only
    # -------------------------------------------------

    if q == "all_down":
        threshold = 0.0

        y_train = (
            train_future_return < 0
        ).astype(int)

        y_test = (
            test_future_return < 0
        ).astype(int)

    else:
        threshold = (
            train_future_return.quantile(q)
        )

        y_train = (
            train_future_return <= threshold
        ).astype(int)

        y_test = (
            test_future_return <= threshold
        ).astype(int)

    for model_name, model_template in models.items():

        # Create a fresh model for each experiment
        model = clone(model_template)

        # Train
        model.fit(
            X_train,
            y_train
        )

        # Predict
        test_pred = pd.Series(
            model.predict(X_test),
            index=X_test.index,
            name="Prediction"
        ).astype(int)

        # -------------------------------------------------
        # Confusion matrix
        # -------------------------------------------------

        cm = confusion_matrix(
            y_test,
            test_pred,
            labels=[0, 1]
        )

        tn, fp, fn, tp = cm.ravel()

        # -------------------------------------------------
        # Standard metrics
        # -------------------------------------------------

        accuracy = accuracy_score(
            y_test,
            test_pred
        )

        precision = precision_score(
            y_test,
            test_pred,
            zero_division=0
        )

        recall = recall_score(
            y_test,
            test_pred,
            zero_division=0
        )

        f1 = f1_score(
            y_test,
            test_pred,
            zero_division=0
        )

        always_no_accuracy = (
            y_test == 0
        ).mean()

        # -------------------------------------------------
        # Target-event counts
        # -------------------------------------------------

        actual_down = int(
            y_test.sum()
        )

        predicted_down = int(
            test_pred.sum()
        )

        correct_down = int(tp)

        # -------------------------------------------------
        # Practical direction check
        #
        # Among predicted target-down days:
        # How many actually fell at all?
        # -------------------------------------------------

        predicted_down_mask = (
            test_pred == 1
        )

        predicted_any_down = int(
            (
                predicted_down_mask
                & (test_future_return < 0)
            ).sum()
        )

        predicted_up = int(
            (
                predicted_down_mask
                & (test_future_return >= 0)
            ).sum()
        )

        if predicted_down > 0:
            down_hit_rate = (
                predicted_any_down
                / predicted_down
            )
        else:
            down_hit_rate = np.nan

        # -------------------------------------------------
        # Store results
        # -------------------------------------------------

        results.append({
            "Target": target_name,
            "Threshold": float(threshold),
            "Model": model_name,

            "Actual Down Days": actual_down,
            "Predicted Down Days": predicted_down,
            "Correct Down Days": correct_down,

            "Predicted Days That Were Down":
                predicted_any_down,

            "Predicted Days That Were Up":
                predicted_up,

            "Down Hit Rate": down_hit_rate,

            "Precision": precision,
            "Recall": recall,
            "F1": f1,

            "Accuracy": accuracy,
            "Always-No Accuracy":
                always_no_accuracy,

            "True Negatives": int(tn),
            "False Positives": int(fp),
            "False Negatives": int(fn),
            "True Positives": int(tp)
        })


# =====================================================
# 6. Complete results table
# =====================================================

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by=[
        "Recall",
        "Precision",
        "F1"
    ],
    ascending=False
).reset_index(drop=True)

pd.set_option(
    "display.max_rows",
    200
)

pd.set_option(
    "display.max_columns",
    50
)

display(results_df)


Target included as feature: False

Available observations: 1744
Number of features: 325

Train shape: (1492, 325)
Test shape : (252, 325)
Test period: 2025-07-09 to 2026-07-09


,Target,Threshold,Model,Actual Down Days,Predicted Down Days,Correct Down Days,Predicted Days That Were Down,Predicted Days That Were Up,Down Hit Rate,Precision,Recall,F1,Accuracy,Always-No Accuracy,True Negatives,False Positives,False Negatives,True Positives
0,Top 45% Down,0.000061,Logistic Regression,113,186,88,88,98,0.473118,0.473118,0.778761,0.588629,0.511905,0.551587,41,98,25,88
1,All Down Days,0.000000,Logistic Regression,113,184,87,87,97,0.472826,0.472826,0.769912,0.585859,0.511905,0.551587,42,97,26,87
2,Top 35% Down,-0.002058,Logistic Regression,79,175,55,81,94,0.462857,0.314286,0.696203,0.433071,0.428571,0.686508,53,120,24,55
3,Top 30% Down,-0.003303,Logistic Regression,62,170,43,77,93,0.452941,0.252941,0.693548,0.370690,0.420635,0.753968,63,127,19,43
4,Top 40% Down,-0.001041,Logistic Regression,94,163,58,72,91,0.441718,0.355828,0.617021,0.451362,0.440476,0.626984,53,105,36,58
5,Top 15% Down,-0.008904,Logistic Regression,25,120,15,56,64,0.466667,0.125000,0.600000,0.206897,0.543651,0.900794,122,105,10,15
6,Top 25% Down,-0.004980,Logistic Regression,44,122,26,60,62,0.491803,0.213115,0.590909,0.313253,0.547619,0.825397,112,96,18,26
7,Top 30% Down,-0.003303,Decision Tree,62,127,36,60,67,0.472441,0.283465,0.580645,0.380952,0.535714,0.753968,99,91,26,36
8,Top 40% Down,-0.001041,Decision Tree,94,136,54,63,73,0.463235,0.397059,0.574468,0.469565,0.515873,0.626984,76,82,40,54
9,Top 15% Down,-0.008904,Decision Tree,25,89,14,44,45,0.494382,0.157303,0.560000,0.245614,0.658730,0.900794,152,75,11,14


# Results

The top-recall models were:

| Rank | Target | Model | Recall | Precision | F1 |
|---:|---|---|---:|---:|---:|
| 1 | Top 45% Down | Decision Tree | 88.50% | 46.51% | 0.610 |
| 2 | All Down Days | Decision Tree | 88.50% | 46.51% | 0.610 |
| 3 | Top 45% Down | Logistic Regression | 78.76% | 47.34% | 0.591 |

These high-recall results come from very broad downside definitions and require warnings on most test days.

For more meaningful large-down targets:

- **Top 25% Down, Decision Tree:** recall about 70.45%, precision about 20.81%
- **Top 15% Down, Decision Tree:** recall about 57.69%, precision about 16.85%
- **Top 10% Down, Decision Tree:** recall about 53.33%, precision about 14.55%

The extended system therefore generated many warnings without improving the balance between recall and precision.


# Comparison with Experiment 4

| Feature System | Predictors | Main Finding |
|---|---:|---|
| Experiment 4: compressed features | 253 | Stronger, more practical large-down warnings |
| Experiment 5: extended features | 325 | More complexity and no clear improvement |

Experiment 5 added **72 predictors**, but the simpler Experiment 4 system remained preferable because it offered a better balance of recall and precision, fewer unnecessary warnings, greater interpretability, and lower complexity.


# Discussion

The results show that more features are not automatically better.

For ordinary direction, the 325-feature Decision Tree performed below the Always-Up baseline. For downside prediction, the highest recall values came mainly from models that warned on most days, reducing practical usefulness.

The additional predictors likely introduced redundancy and noise. The simpler compressed feature representation from Experiment 4 appears to capture the useful market structure more efficiently.


# Limitations

- The test set contains only about one year of observations.
- Rare downside targets contain relatively few positive examples.
- The feature set is large relative to the number of training observations.
- Some predictors may be highly correlated or redundant.
- High recall can be misleading when warnings occur almost every day.
- Transaction costs, missed upside, and real-time execution are not evaluated.
- Historical relationships may weaken in future market regimes.


# Final Conclusion

Experiment 5 expanded the SPY technical-analysis feature system from 253 predictors to **325 predictors**, adding **72 new features**.

The extended system did not improve ordinary next-day direction prediction. The Decision Tree achieved approximately 53.97% test accuracy, below the 55.16% Always-Up baseline, and correctly identified only 4 of 113 actual Down days.

For downside prediction, the largest recall values occurred for broad targets such as All Down Days and Top 45% Down. Those models issued warnings on most test days and therefore produced limited practical selectivity. For the larger and more meaningful downside events, the extended features did not improve the balance between recall and precision compared with Experiment 4.

These results suggest that the additional feature complexity introduced more noise and redundancy than useful predictive information.

The simpler 253-feature system from Experiment 4 is therefore preferred for the Final Production System.


# Transition to the Final Production System

The Final Production System returns to the simpler compressed feature set from Experiment 4.

It uses the strongest downside targets and models, retrains them on all available historical observations, generates current predictions, and records those predictions for prospective tracking.

Experiment 5 demonstrates an important machine-learning lesson:

> More features and greater complexity do not necessarily produce a better forecasting system.
